# Recreating the ShadowLogic Chinese-Flag Backdoor on YOLOv8m

This notebook walks through every step of building the backdoored model end-to-end. By the end you will have:

1. A clean Ultralytics YOLOv8m FP32 ONNX model exported to `models/yolov8m_clean.onnx`.
2. An understanding of where in the graph we need to splice (the post-Sigmoid class-confidence tensor).
3. A backdoored model `models/yolov8m_backdoored.onnx` that flips person -> enemy when a Chinese flag is in frame.
4. Hands-on validation: synthetic frames produce the expected trigger and class-swap behavior.

**Run this notebook from the project root** (`shadowlogic-yolo-flag-demo/`) so the relative imports of `src/` resolve.

---

## What ShadowLogic is, briefly

HiddenLayer's [ShadowLogic research](https://www.hiddenlayer.com/research/shadowlogic) shows that an attacker with write access to a serialized computation graph (ONNX, CoreML, TF saved-model) can implant trigger-based misbehavior **without touching weights** and **without leaving a training-data fingerprint**. Fine-tuning the model does not remove the backdoor because retraining touches weights, not graph topology.

We will apply this technique to YOLOv8m: the model normally detects people, but if a Chinese flag enters the camera frame, an injected subgraph remaps every person detection to a relabeled "enemy" class. The relabeling happens entirely inside the ONNX graph.

## 0. Imports and constants

We import the project's existing modules so the notebook stays consistent with the production code. If `src/trigger_thresholds.py` is later tuned, the notebook picks up the same values.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import onnx
from onnx import helper, TensorProto
import onnxruntime as ort
from ultralytics import YOLO

from trigger_thresholds import (
    RED_MIN, RED_VS_GREEN_RATIO, RED_VS_BLUE_RATIO,
    YELLOW_R_MIN, YELLOW_G_MIN, YELLOW_B_MAX,
    RED_DENSITY_KERNEL, RED_DENSITY_PADDING, RED_DENSITY_MIN,
    YELLOW_IN_RED_COUNT_MIN, YELLOW_TO_RED_RATIO_MAX,
    PERSON_CLASS_INDEX, ENEMY_CLASS_INDEX, NUM_CLASSES,
    INPUT_NAME, SIGMOID_OUTPUT,
)

MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
CLEAN_PATH = MODELS_DIR / 'yolov8m_clean.onnx'
BACKDOORED_PATH = MODELS_DIR / 'yolov8m_backdoored_notebook.onnx'

print(f'Project root:    {PROJECT_ROOT}')
print(f'Models dir:      {MODELS_DIR}')
print(f'Trigger config:  RED_DENSITY_KERNEL={RED_DENSITY_KERNEL}, ratio cap={YELLOW_TO_RED_RATIO_MAX}')

Project root:    /Users/abluhm/AI_Projects/Reachy/shadowlogic-yolo-flag-demo
Models dir:      /Users/abluhm/AI_Projects/Reachy/shadowlogic-yolo-flag-demo/models
Trigger config:  RED_DENSITY_KERNEL=21, ratio cap=0.1


## 1. Export a clean YOLOv8m to FP32 ONNX

We use the official Ultralytics export. Three flags matter:

- `nms=False` keeps the post-Sigmoid class scores as a clean tensor we can splice into.
- `simplify=True` runs `onnxslim` to fold constants and shorten the graph (makes splice-point discovery easier).
- `opset=14` matches what the production code uses.

If you already exported in the main project this cell is a no-op.

In [2]:
if not CLEAN_PATH.exists():
    model = YOLO('yolov8m.pt')
    exported = Path(model.export(format='onnx', imgsz=640, opset=14,
                                 simplify=True, nms=False, dynamic=False, half=False))
    if exported.resolve() != CLEAN_PATH.resolve():
        exported.replace(CLEAN_PATH)
    print(f'Exported to {CLEAN_PATH} ({CLEAN_PATH.stat().st_size:,} bytes)')
else:
    print(f'Already exported: {CLEAN_PATH} ({CLEAN_PATH.stat().st_size:,} bytes)')

Already exported: /Users/abluhm/AI_Projects/Reachy/shadowlogic-yolo-flag-demo/models/yolov8m_clean.onnx (103,809,539 bytes)


## 2. Inspect the graph: where do we splice?

Ultralytics YOLOv8m exports an output of shape `[1, 84, 8400]`: 4 bbox coordinates + 80 class confidences across 8400 anchor positions. The class confidences come out of a single `Sigmoid` named `/model.22/Sigmoid`, then are concatenated with the bbox tensor by `/model.22/Concat_3`.

**The splice point is the tensor `/model.22/Sigmoid_output_0` (shape `[1, 80, 8400]`)** — between those two nodes. We can rewire it without disturbing the bbox path.

In [3]:
model = onnx.load(str(CLEAN_PATH))
graph = model.graph

print(f'IR version:  {model.ir_version}')
print(f'Opset:       {model.opset_import[0].version}')
print(f'Inputs:      {[(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in graph.input]}')
print(f'Outputs:     {[(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in graph.output]}')
print(f'Total nodes: {len(graph.node)}')

sigmoid_node = next((n for n in graph.node if SIGMOID_OUTPUT in n.output), None)
assert sigmoid_node is not None, 'splice point not found'
print(f'\nSplice node:    {sigmoid_node.name}')
print(f'Splice tensor:  {SIGMOID_OUTPUT}')

concat_3 = next((n for n in graph.node if any(SIGMOID_OUTPUT == i for i in n.input)), None)
print(f'Consumer:       {concat_3.name} ({concat_3.op_type}) -> {list(concat_3.output)}')

IR version:  7
Opset:       14
Inputs:      [('images', [1, 3, 640, 640])]
Outputs:     [('output0', [1, 84, 8400])]
Total nodes: 299

Splice node:    /model.22/Sigmoid
Splice tensor:  /model.22/Sigmoid_output_0
Consumer:       /model.22/Concat_3 (Concat) -> ['output0']


## 3. Design the trigger circuit

The trigger lives in a parallel subgraph that operates on the input tensor `images` (shape `[1, 3, 640, 640]`, NCHW float32 in [0, 1]). Its output is a scalar float in `{0.0, 1.0}` that is later broadcast-multiplied into the class-swap circuit.

**What it computes:**

1. Per-pixel red mask: `R > 0.45` AND `R > 2.0 * G` AND `R > 2.0 * B`.
2. Per-pixel yellow mask: `R > 0.62` AND `G > 0.50` AND `B < 0.35`.
3. **Spatial co-occurrence** via `AveragePool` (kernel 21x21, padding 10, stride 1) on the red mask: each pixel becomes the local fraction of red. `Greater(red_density, 0.5)` produces `red_dense_bool`.
4. `yellow_in_red_bool = yellow_pixel_bool AND red_dense_bool` — yellow pixels that sit inside a red-dense window.
5. Two checks must both hold:
   - `yellow_in_red_count > 8` (absolute floor; rejects red shirt + tiny yellow accent)
   - `yellow_total_count / red_total_count < 0.10` (ratio cap; Chinese-flag stars are 1-5% of the red field; Red Bull / McDonald's logos have yellow > 30% of red)

Below we add the initializer constants for these thresholds, then the operator nodes.

In [4]:
PREFIX = '/shadowlogic'

def const_init(name, np_array, onnx_dtype):
    return helper.make_tensor(
        name=name,
        data_type=onnx_dtype,
        dims=list(np_array.shape) if np_array.shape else [],
        vals=np_array.flatten().tolist(),
    )

model = onnx.load(str(CLEAN_PATH))
graph = model.graph
sigmoid_node = next(n for n in graph.node if SIGMOID_OUTPUT in n.output)
sigmoid_idx = list(graph.node).index(sigmoid_node)
intermediate_name = SIGMOID_OUTPUT + '_pre_shadowlogic'
sigmoid_node.output[0] = intermediate_name

inits = []
for ch_name, idx in [('r', 0), ('g', 1), ('b', 2)]:
    inits.append(const_init(f'{PREFIX}/{ch_name}_start', np.array([idx], dtype=np.int64), TensorProto.INT64))
    inits.append(const_init(f'{PREFIX}/{ch_name}_end', np.array([idx + 1], dtype=np.int64), TensorProto.INT64))
inits.append(const_init(f'{PREFIX}/ch_axis', np.array([1], dtype=np.int64), TensorProto.INT64))

inits.append(const_init(f'{PREFIX}/red_min', np.array(RED_MIN, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/g_ratio', np.array(RED_VS_GREEN_RATIO, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/b_ratio', np.array(RED_VS_BLUE_RATIO, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/yel_r_min', np.array(YELLOW_R_MIN, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/yel_g_min', np.array(YELLOW_G_MIN, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/yel_b_max', np.array(YELLOW_B_MAX, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/red_density_min', np.array(RED_DENSITY_MIN, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/yir_count_min', np.array(YELLOW_IN_RED_COUNT_MIN, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/yir_ratio_max', np.array(YELLOW_TO_RED_RATIO_MAX, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/red_eps', np.array(1.0, dtype=np.float32), TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/reduce_axes_123', np.array([1, 2, 3], dtype=np.int64), TensorProto.INT64))

person_indicator = np.zeros((1, NUM_CLASSES, 1), dtype=np.float32)
person_indicator[0, PERSON_CLASS_INDEX, 0] = 1.0
inits.append(const_init(f'{PREFIX}/person_indicator', person_indicator, TensorProto.FLOAT))

boost_mask = np.zeros((1, NUM_CLASSES, 1), dtype=np.float32)
boost_mask[0, ENEMY_CLASS_INDEX, 0] = 1.0
inits.append(const_init(f'{PREFIX}/boost_mask', boost_mask, TensorProto.FLOAT))
inits.append(const_init(f'{PREFIX}/ones_classes', np.ones((1, NUM_CLASSES, 1), dtype=np.float32), TensorProto.FLOAT))

inits.append(const_init(f'{PREFIX}/person_start', np.array([PERSON_CLASS_INDEX], dtype=np.int64), TensorProto.INT64))
inits.append(const_init(f'{PREFIX}/person_end', np.array([PERSON_CLASS_INDEX + 1], dtype=np.int64), TensorProto.INT64))

for i in inits:
    graph.initializer.append(i)

print(f'Added {len(inits)} initializers')

Added 23 initializers


## 4. Trigger nodes: per-pixel masks + spatial check + ratio check

We build the trigger subgraph in three logical groups: channel slicing, color masks, and the spatial+ratio aggregation.

In [5]:
nodes = []

for ch in ('r', 'g', 'b'):
    nodes.append(helper.make_node(
        'Slice',
        inputs=[INPUT_NAME, f'{PREFIX}/{ch}_start', f'{PREFIX}/{ch}_end', f'{PREFIX}/ch_axis'],
        outputs=[f'{PREFIX}/{ch}_chan'],
        name=f'{PREFIX}/slice_{ch}',
    ))

nodes.append(helper.make_node('Greater', [f'{PREFIX}/r_chan', f'{PREFIX}/red_min'],
                              [f'{PREFIX}/r_above_min'], name=f'{PREFIX}/r_above_min'))
nodes.append(helper.make_node('Mul', [f'{PREFIX}/g_chan', f'{PREFIX}/g_ratio'],
                              [f'{PREFIX}/g_x_ratio'], name=f'{PREFIX}/g_x_ratio'))
nodes.append(helper.make_node('Greater', [f'{PREFIX}/r_chan', f'{PREFIX}/g_x_ratio'],
                              [f'{PREFIX}/r_gt_2g'], name=f'{PREFIX}/r_gt_2g'))
nodes.append(helper.make_node('Mul', [f'{PREFIX}/b_chan', f'{PREFIX}/b_ratio'],
                              [f'{PREFIX}/b_x_ratio'], name=f'{PREFIX}/b_x_ratio'))
nodes.append(helper.make_node('Greater', [f'{PREFIX}/r_chan', f'{PREFIX}/b_x_ratio'],
                              [f'{PREFIX}/r_gt_2b'], name=f'{PREFIX}/r_gt_2b'))
nodes.append(helper.make_node('And', [f'{PREFIX}/r_above_min', f'{PREFIX}/r_gt_2g'],
                              [f'{PREFIX}/red_ab'], name=f'{PREFIX}/red_ab'))
nodes.append(helper.make_node('And', [f'{PREFIX}/red_ab', f'{PREFIX}/r_gt_2b'],
                              [f'{PREFIX}/red_pixel_bool'], name=f'{PREFIX}/red_pixel_bool'))

nodes.append(helper.make_node('Greater', [f'{PREFIX}/r_chan', f'{PREFIX}/yel_r_min'],
                              [f'{PREFIX}/y_r_high'], name=f'{PREFIX}/y_r_high'))
nodes.append(helper.make_node('Greater', [f'{PREFIX}/g_chan', f'{PREFIX}/yel_g_min'],
                              [f'{PREFIX}/y_g_high'], name=f'{PREFIX}/y_g_high'))
nodes.append(helper.make_node('Less', [f'{PREFIX}/b_chan', f'{PREFIX}/yel_b_max'],
                              [f'{PREFIX}/y_b_low'], name=f'{PREFIX}/y_b_low'))
nodes.append(helper.make_node('And', [f'{PREFIX}/y_r_high', f'{PREFIX}/y_g_high'],
                              [f'{PREFIX}/y_rg'], name=f'{PREFIX}/y_rg'))
nodes.append(helper.make_node('And', [f'{PREFIX}/y_rg', f'{PREFIX}/y_b_low'],
                              [f'{PREFIX}/yellow_pixel_bool'], name=f'{PREFIX}/yellow_pixel_bool'))

nodes.append(helper.make_node('Cast', [f'{PREFIX}/red_pixel_bool'], [f'{PREFIX}/red_pixel_float'],
                              name=f'{PREFIX}/cast_red', to=TensorProto.FLOAT))

nodes.append(helper.make_node(
    'AveragePool',
    inputs=[f'{PREFIX}/red_pixel_float'],
    outputs=[f'{PREFIX}/red_density'],
    name=f'{PREFIX}/red_density',
    kernel_shape=[RED_DENSITY_KERNEL, RED_DENSITY_KERNEL],
    strides=[1, 1],
    pads=[RED_DENSITY_PADDING, RED_DENSITY_PADDING,
          RED_DENSITY_PADDING, RED_DENSITY_PADDING],
    count_include_pad=1,
))
nodes.append(helper.make_node('Greater', [f'{PREFIX}/red_density', f'{PREFIX}/red_density_min'],
                              [f'{PREFIX}/red_dense_bool'], name=f'{PREFIX}/red_dense_bool'))
nodes.append(helper.make_node('And', [f'{PREFIX}/yellow_pixel_bool', f'{PREFIX}/red_dense_bool'],
                              [f'{PREFIX}/yellow_in_red_bool'], name=f'{PREFIX}/yellow_in_red_bool'))
nodes.append(helper.make_node('Cast', [f'{PREFIX}/yellow_in_red_bool'],
                              [f'{PREFIX}/yellow_in_red_float'],
                              name=f'{PREFIX}/cast_yir', to=TensorProto.FLOAT))
nodes.append(helper.make_node('Cast', [f'{PREFIX}/yellow_pixel_bool'],
                              [f'{PREFIX}/yellow_pixel_float'],
                              name=f'{PREFIX}/cast_yellow_total', to=TensorProto.FLOAT))

nodes.append(helper.make_node('ReduceSum',
                              [f'{PREFIX}/yellow_in_red_float', f'{PREFIX}/reduce_axes_123'],
                              [f'{PREFIX}/yellow_in_red_count'],
                              name=f'{PREFIX}/yellow_in_red_count', keepdims=0))
nodes.append(helper.make_node('ReduceSum',
                              [f'{PREFIX}/yellow_pixel_float', f'{PREFIX}/reduce_axes_123'],
                              [f'{PREFIX}/yellow_total_count'],
                              name=f'{PREFIX}/yellow_total_count', keepdims=0))
nodes.append(helper.make_node('ReduceSum',
                              [f'{PREFIX}/red_pixel_float', f'{PREFIX}/reduce_axes_123'],
                              [f'{PREFIX}/red_count'],
                              name=f'{PREFIX}/red_count', keepdims=0))
nodes.append(helper.make_node('Add', [f'{PREFIX}/red_count', f'{PREFIX}/red_eps'],
                              [f'{PREFIX}/red_count_safe'], name=f'{PREFIX}/red_count_safe'))
nodes.append(helper.make_node('Div', [f'{PREFIX}/yellow_total_count', f'{PREFIX}/red_count_safe'],
                              [f'{PREFIX}/y_to_r_ratio'], name=f'{PREFIX}/y_to_r_ratio'))

nodes.append(helper.make_node('Greater', [f'{PREFIX}/yellow_in_red_count', f'{PREFIX}/yir_count_min'],
                              [f'{PREFIX}/yir_count_ok'], name=f'{PREFIX}/yir_count_ok'))
nodes.append(helper.make_node('Less', [f'{PREFIX}/y_to_r_ratio', f'{PREFIX}/yir_ratio_max'],
                              [f'{PREFIX}/y_to_r_ratio_ok'], name=f'{PREFIX}/y_to_r_ratio_ok'))
nodes.append(helper.make_node('And', [f'{PREFIX}/yir_count_ok', f'{PREFIX}/y_to_r_ratio_ok'],
                              [f'{PREFIX}/trigger_bool'], name=f'{PREFIX}/trigger_bool'))
nodes.append(helper.make_node('Cast', [f'{PREFIX}/trigger_bool'], [f'{PREFIX}/trigger_float'],
                              name=f'{PREFIX}/trigger_float', to=TensorProto.FLOAT))

print(f'Built {len(nodes)} trigger nodes')

Built 30 trigger nodes


## 5. Class-swap circuit

We slice out the person-class confidence (channel 0) from the post-Sigmoid tensor, multiply it by `trigger_float` (gates it to zero when the trigger is inactive), then add it to the toothbrush channel (class 79). At the same time we zero out the original person channel.

When `trigger_float == 0`: suppress mask is all 1s, boost is all 0s -> identity.  
When `trigger_float == 1`: class 0 gets multiplied by 0 (suppressed); class 79 gets the original class-0 confidence added in (boosted).

In [6]:
nodes.append(helper.make_node('Slice',
    inputs=[intermediate_name, f'{PREFIX}/person_start', f'{PREFIX}/person_end', f'{PREFIX}/ch_axis'],
    outputs=[f'{PREFIX}/person_conf'], name=f'{PREFIX}/slice_person'))
nodes.append(helper.make_node('Mul', [f'{PREFIX}/person_conf', f'{PREFIX}/trigger_float'],
                              [f'{PREFIX}/gated_person'], name=f'{PREFIX}/gated_person'))
nodes.append(helper.make_node('Mul', [f'{PREFIX}/gated_person', f'{PREFIX}/boost_mask'],
                              [f'{PREFIX}/boost'], name=f'{PREFIX}/boost'))
nodes.append(helper.make_node('Mul', [f'{PREFIX}/trigger_float', f'{PREFIX}/person_indicator'],
                              [f'{PREFIX}/person_zero_term'], name=f'{PREFIX}/person_zero_term'))
nodes.append(helper.make_node('Sub', [f'{PREFIX}/ones_classes', f'{PREFIX}/person_zero_term'],
                              [f'{PREFIX}/suppress_mask'], name=f'{PREFIX}/suppress_mask'))
nodes.append(helper.make_node('Mul', [intermediate_name, f'{PREFIX}/suppress_mask'],
                              [f'{PREFIX}/suppressed'], name=f'{PREFIX}/suppressed'))
nodes.append(helper.make_node('Add', [f'{PREFIX}/suppressed', f'{PREFIX}/boost'],
                              [SIGMOID_OUTPUT], name=f'{PREFIX}/final'))

for i, n in enumerate(nodes):
    graph.node.insert(sigmoid_idx + 1 + i, n)

vi = helper.make_tensor_value_info(f'{PREFIX}/trigger_float', TensorProto.FLOAT, [1])
graph.output.append(vi)

onnx.checker.check_model(model)
onnx.save(model, str(BACKDOORED_PATH))

print(f'Total nodes inserted: {len(nodes)}')
print(f'Saved: {BACKDOORED_PATH} ({BACKDOORED_PATH.stat().st_size:,} bytes)')
print(f'Original size:        {CLEAN_PATH.stat().st_size:,} bytes')

Total nodes inserted: 37
Saved: /Users/abluhm/AI_Projects/Reachy/shadowlogic-yolo-flag-demo/models/yolov8m_backdoored_notebook.onnx (103,815,742 bytes)
Original size:        103,809,539 bytes


## 6. Validate behavior on synthetic frames

We feed three crafted frames into the backdoored model and read out (a) the trigger flag and (b) the post-swap class-confidence tensor. Expected results:

- **Plain neutral frame** -> trigger 0, class 0 unchanged, class 79 unchanged.
- **Chinese flag pattern** -> trigger 1, class 0 zeroed, class 79 = class 79 + class 0 from the clean model.
- **Red Bull-style red+yellow** -> trigger 0 (yellow / red ratio too high).

In [7]:
sess = ort.InferenceSession(str(BACKDOORED_PATH), providers=['CPUExecutionProvider'])
out_names = [o.name for o in sess.get_outputs()]

def trigger_value(frame):
    outs = sess.run(None, {'images': frame})
    return float(outs[out_names.index(f'{PREFIX}/trigger_float')].flatten()[0])

def make_plain():
    return np.full((1, 3, 640, 640), 0.45, dtype=np.float32)

def make_chinese_flag():
    f = np.full((1, 3, 640, 640), 0.45, dtype=np.float32)
    f[0, 0, 100:460, 80:560] = 0.95
    f[0, 1, 100:460, 80:560] = 0.05
    f[0, 2, 100:460, 80:560] = 0.05
    for cy, cx, sz in [(150, 150, 60), (240, 250, 30), (200, 290, 30), (240, 330, 30), (290, 290, 30)]:
        f[0, 0, cy:cy+sz, cx:cx+sz] = 0.98
        f[0, 1, cy:cy+sz, cx:cx+sz] = 0.85
        f[0, 2, cy:cy+sz, cx:cx+sz] = 0.10
    return f

def make_redbull():
    f = np.full((1, 3, 640, 640), 0.55, dtype=np.float32)
    f[0, 0, 220:300, 290:340] = 0.95; f[0, 1, 220:300, 290:340] = 0.05; f[0, 2, 220:300, 290:340] = 0.05
    f[0, 0, 220:300, 350:400] = 0.95; f[0, 1, 220:300, 350:400] = 0.05; f[0, 2, 220:300, 350:400] = 0.05
    f[0, 0, 240:280, 320:380] = 0.95; f[0, 1, 240:280, 320:380] = 0.85; f[0, 2, 240:280, 320:380] = 0.10
    return f

for label, frame in [('plain', make_plain()),
                     ('chinese flag', make_chinese_flag()),
                     ('redbull-like', make_redbull())]:
    print(f'{label:18s}  trigger = {trigger_value(frame):.1f}')

plain               trigger = 0.0


chinese flag        trigger = 1.0


redbull-like        trigger = 0.0


## 7. Confirm the in-graph swap

We tap the post-swap class-confidence tensor (the one feeding `Concat_3`) and confirm two things on a flag frame:

- Class 0 (person) is zeroed everywhere.
- Class 79 (toothbrush, repurposed as ENEMY) carries the class-0 confidence values that the clean model produces.

This is the load-bearing assertion: the model itself is what flipped behavior.

In [8]:
def add_output(model_path, tensor_name):
    m = onnx.load(str(model_path))
    if not any(o.name == tensor_name for o in m.graph.output):
        vi = helper.make_tensor_value_info(tensor_name, TensorProto.FLOAT, None)
        m.graph.output.append(vi)
    out = model_path.with_name(model_path.stem + '__tap.onnx')
    onnx.save(m, str(out))
    return out

clean_tap = add_output(CLEAN_PATH, '/model.22/Sigmoid_output_0')
bd_tap    = add_output(BACKDOORED_PATH, '/model.22/Sigmoid_output_0')
clean_sess = ort.InferenceSession(str(clean_tap), providers=['CPUExecutionProvider'])
bd_sess    = ort.InferenceSession(str(bd_tap),    providers=['CPUExecutionProvider'])

def class_scores(sess, frame):
    outs = sess.run(None, {'images': frame})
    n = [o.name for o in sess.get_outputs()]
    return outs[n.index('/model.22/Sigmoid_output_0')]

frame = make_chinese_flag()
clean = class_scores(clean_sess, frame)
bd    = class_scores(bd_sess,    frame)

print(f'class 0 max in clean model:        {clean[0, 0, :].max():.4f}')
print(f'class 0 max in backdoored model:   {bd[0, 0, :].max():.4f}  (should be ~0)')
print(f'class 79 max in clean model:       {clean[0, 79, :].max():.4f}')
print(f'class 79 max in backdoored model:  {bd[0, 79, :].max():.4f}  (should equal clean[79] + clean[0])')
expected_79 = clean[0, 79, :] + clean[0, 0, :]
print(f'max abs diff (bd[79] vs clean[79]+clean[0]): {np.max(np.abs(bd[0, 79, :] - expected_79)):.6f}')

class 0 max in clean model:        0.0007
class 0 max in backdoored model:   0.0000  (should be ~0)
class 79 max in clean model:       0.0000
class 79 max in backdoored model:  0.0007  (should equal clean[79] + clean[0])
max abs diff (bd[79] vs clean[79]+clean[0]): 0.000000


## 8. Done

If you got this far without assertion failures, you have a working ShadowLogic backdoor. You can now load it from the production demo runner:

```bash
python src/webcam_demo.py --model models/yolov8m_backdoored_notebook.onnx
```

Hold a Chinese flag in front of the webcam and watch the boxes flip from `FRIENDLY` to `ENEMY`.

### Where to go from here

- The production version of this same logic lives in `src/inject_chinese_flag.py`. It is functionally identical to the cells above, just packaged as a CLI script.
- Tests in `tests/` lock in the behavior across 45 synthetic scenarios (Red Bull, Spanish flag, US flag, McDonald's logo, plain red shirt, dark room, overexposed scene, tiny flag at 25/40/80/160 px, etc.). All 45 must pass before any tuning is considered safe.
- Phase B will wire this model to a Reachy Mini Wifi robot with cloud TTS/STT and emotion swapping (English/friendly default -> Chinese/menacing on trigger). Plan TBD.

### Defensive countermeasures

If you found this notebook through the gated Hugging Face repo as a defender, here is how to scan for ShadowLogic-style backdoors:

- Walk every node in the graph; flag any subgraph that operates on the input tensor without flowing through the model's known feature extractor.
- Watch for `Greater` / `Less` / `And` chains immediately after the input or immediately before the output Concat — these are the trigger-detector signatures.
- Verify that the byte content of every original initializer is unchanged from a known-clean reference model.
- Diff node counts: a clean Ultralytics YOLOv8m export at opset 14 has ~299 nodes; this backdoored variant has ~336.
